In [1]:
import pandas as pd
import numpy as np

In [2]:
telco_data = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(telco_data.shape)

(7043, 21)


In [4]:
telco_data["TotalCharges"] = pd.to_numeric(
    telco_data["TotalCharges"],
    errors="coerce"
)

# Fill missing TotalCharges
telco_data["TotalCharges"] = telco_data["TotalCharges"].fillna(
    telco_data["TotalCharges"].median()
)
telco_data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


#### Tenure groups

In [6]:
telco_data["TenureGroup"] = pd.cut(
    telco_data["tenure"],
    bins=[-1, 12, 24, 48, 60, 72],
    labels=[
        "New Customer",
        "Short Term",
        "Medium Term",
        "Long Term",
        "Very Long Term"
    ]
)

telco_data["TenureGroup"].value_counts()

TenureGroup
New Customer      2186
Medium Term       1594
Very Long Term    1407
Short Term        1024
Long Term          832
Name: count, dtype: int64

#### Number of Services Feature

In [7]:
service_columns = [
    "PhoneService",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]


In [8]:
telco_data["NumberOfServices"] = 0

for column in service_columns:
    for i in range(len(telco_data)):
        value = telco_data.loc[i, column]

        if value != "No" and value != "No internet service":
            telco_data.loc[i, "NumberOfServices"] += 1

print(telco_data["NumberOfServices"].value_counts().sort_index())

NumberOfServices
1    1667
2    1158
3     957
4     978
5     933
6     722
7     420
8     208
Name: count, dtype: int64


In [9]:
telco_data[["customerID", "NumberOfServices"]].head(10)

,customerID,NumberOfServices
0,7590-VHVEG,2
1,5575-GNVDE,3
2,3668-QPYBK,3
3,7795-CFOCW,4
4,9237-HQITU,1
5,9305-CDSKC,5
6,1452-KIOVK,4
7,6713-OKOMC,2
8,7892-POOKP,6
9,6388-TABGU,3


#### Internet Service Indicators

In [10]:
telco_data["IsDSL"] = 0
telco_data["IsFiberOptic"] = 0
telco_data["NoInternet"] = 0

for i in range(len(telco_data)):
    if telco_data.loc[i, "InternetService"] == "DSL":
        telco_data.loc[i, "IsDSL"] = 1
    elif telco_data.loc[i, "InternetService"] == "Fiber optic":
        telco_data.loc[i, "IsFiberOptic"] = 1
    elif telco_data.loc[i, "InternetService"] == "No":
        telco_data.loc[i, "NoInternet"] = 1

print(telco_data[[
    "InternetService",
    "IsDSL",
    "IsFiberOptic",
    "NoInternet"
]].head(10))

  InternetService  IsDSL  IsFiberOptic  NoInternet
0             DSL      1             0           0
1             DSL      1             0           0
2             DSL      1             0           0
3             DSL      1             0           0
4     Fiber optic      0             1           0
5     Fiber optic      0             1           0
6     Fiber optic      0             1           0
7             DSL      1             0           0
8     Fiber optic      0             1           0
9             DSL      1             0           0


#### Monthly Charges Group

In [11]:
telco_data["MonthlyChargesGroup"] = ""

for i in range(len(telco_data)):
    charge = telco_data.loc[i, "MonthlyCharges"]

    if charge < 35:
        telco_data.loc[i, "MonthlyChargesGroup"] = "Low"
    elif charge < 70:
        telco_data.loc[i, "MonthlyChargesGroup"] = "Medium"
    else:
        telco_data.loc[i, "MonthlyChargesGroup"] = "High"

print(telco_data["MonthlyChargesGroup"].value_counts())

MonthlyChargesGroup
High      3591
Low       1731
Medium    1721
Name: count, dtype: int64


In [12]:
telco_data[["MonthlyCharges", "MonthlyChargesGroup"]].head(10)

,MonthlyCharges,MonthlyChargesGroup
0,29.85,Low
1,56.95,Medium
2,53.85,Medium
3,42.30,Medium
4,70.70,High
5,99.65,High
6,89.10,High
7,29.75,Low
8,104.80,High
9,56.15,Medium


#### Total Charges Group

In [13]:
telco_data["TotalChargesGroup"] = ""

for i in range(len(telco_data)):
    charge = telco_data.loc[i, "TotalCharges"]

    if charge < 1000:
        telco_data.loc[i, "TotalChargesGroup"] = "Low"
    elif charge < 3000:
        telco_data.loc[i, "TotalChargesGroup"] = "Medium"
    else:
        telco_data.loc[i, "TotalChargesGroup"] = "High"

print(telco_data["TotalChargesGroup"].value_counts())

TotalChargesGroup
Low       2893
High      2204
Medium    1946
Name: count, dtype: int64


In [14]:
telco_data[["TotalCharges", "TotalChargesGroup"]].head(10)

,TotalCharges,TotalChargesGroup
0,29.85,Low
1,1889.50,Medium
2,108.15,Low
3,1840.75,Medium
4,151.65,Low
5,820.50,Low
6,1949.40,Medium
7,301.90,Low
8,3046.05,High
9,3487.95,High


#### Charge-to-Tenure Ratio

In [15]:
telco_data["ChargeToTenureRatio"] = 0.0

for i in range(len(telco_data)):
    tenure = telco_data.loc[i, "tenure"]
    total_charge = telco_data.loc[i, "TotalCharges"]

    if tenure > 0:
        telco_data.loc[i, "ChargeToTenureRatio"] = total_charge / tenure
    else:
        telco_data.loc[i, "ChargeToTenureRatio"] = 0

print(telco_data[["tenure", "TotalCharges", "ChargeToTenureRatio"]].head(10))

   tenure  TotalCharges  ChargeToTenureRatio
0       1         29.85            29.850000
1      34       1889.50            55.573529
2       2        108.15            54.075000
3      45       1840.75            40.905556
4       2        151.65            75.825000
5       8        820.50           102.562500
6      22       1949.40            88.609091
7      10        301.90            30.190000
8      28       3046.05           108.787500
9      62       3487.95            56.257258


#### High-Value Customer Indicator

In [16]:
telco_data["HighValueCustomer"] = 0

for i in range(len(telco_data)):
    monthly_charge = telco_data.loc[i, "MonthlyCharges"]
    total_charge = telco_data.loc[i, "TotalCharges"]

    if monthly_charge >= 70 and total_charge >= 3000:
        telco_data.loc[i, "HighValueCustomer"] = 1

print(telco_data["HighValueCustomer"].value_counts())

HighValueCustomer
0    5153
1    1890
Name: count, dtype: int64


In [17]:
telco_data[["MonthlyCharges", "TotalCharges", "HighValueCustomer"]].head(10)

,MonthlyCharges,TotalCharges,HighValueCustomer
0,29.85,29.85,0
1,56.95,1889.50,0
2,53.85,108.15,0
3,42.30,1840.75,0
4,70.70,151.65,0
5,99.65,820.50,0
6,89.10,1949.40,0
7,29.75,301.90,0
8,104.80,3046.05,1
9,56.15,3487.95,0


#### High-Risk Customer Indicator

In [18]:
telco_data["HighRiskCustomer"] = 0

for i in range(len(telco_data)):
    tenure = telco_data.loc[i, "tenure"]
    contract = telco_data.loc[i, "Contract"]
    monthly_charge = telco_data.loc[i, "MonthlyCharges"]

    if tenure <= 12 and contract == "Month-to-month" and monthly_charge >= 70:
        telco_data.loc[i, "HighRiskCustomer"] = 1

print(telco_data["HighRiskCustomer"].value_counts())

HighRiskCustomer
0    6183
1     860
Name: count, dtype: int64


In [19]:
telco_data[["tenure", "Contract", "MonthlyCharges", "HighRiskCustomer"]].head(10)

,tenure,Contract,MonthlyCharges,HighRiskCustomer
0,1,Month-to-month,29.85,0
1,34,One year,56.95,0
2,2,Month-to-month,53.85,0
3,45,One year,42.30,0
4,2,Month-to-month,70.70,1
5,8,Month-to-month,99.65,1
6,22,Month-to-month,89.10,0
7,10,Month-to-month,29.75,0
8,28,Month-to-month,104.80,0
9,62,One year,56.15,0


#### Service Combination Feature

In [21]:
telco_data["InternetTechSupport"] = ""

for i in range(len(telco_data)):
    internet = telco_data.loc[i, "InternetService"]
    support = telco_data.loc[i, "TechSupport"]

    if internet != "No" and support == "Yes":
        telco_data.loc[i, "InternetTechSupport"] = "Internet_With_Support"
    elif internet != "No" and support != "Yes":
        telco_data.loc[i, "InternetTechSupport"] = "Internet_Without_Support"
    else:
        telco_data.loc[i, "InternetTechSupport"] = "No_Internet"

print(telco_data["InternetTechSupport"].value_counts())

InternetTechSupport
Internet_Without_Support    3473
Internet_With_Support       2044
No_Internet                 1526
Name: count, dtype: int64


In [22]:
engineered_features = [
    "TenureGroup",
    "NumberOfServices",
    "IsDSL",
    "IsFiberOptic",
    "NoInternet",
    "MonthlyChargesGroup",
    "TotalChargesGroup",
    "ChargeToTenureRatio",
    "HighValueCustomer",
    "HighRiskCustomer",
    "InternetTechSupport"
]

In [24]:
telco_data[engineered_features].head(10)

,TenureGroup,NumberOfServices,IsDSL,IsFiberOptic,NoInternet,MonthlyChargesGroup,TotalChargesGroup,ChargeToTenureRatio,HighValueCustomer,HighRiskCustomer,InternetTechSupport
0,New Customer,2,1,0,0,Low,Low,29.850000,0,0,Internet_Without_Support
1,Medium Term,3,1,0,0,Medium,Medium,55.573529,0,0,Internet_Without_Support
2,New Customer,3,1,0,0,Medium,Low,54.075000,0,0,Internet_Without_Support
3,Medium Term,4,1,0,0,Medium,Medium,40.905556,0,0,Internet_With_Support
4,New Customer,1,0,1,0,High,Low,75.825000,0,1,Internet_Without_Support
5,New Customer,5,0,1,0,High,Low,102.562500,0,1,Internet_Without_Support
6,Short Term,4,0,1,0,High,Medium,88.609091,0,0,Internet_Without_Support
7,New Customer,2,1,0,0,Low,Low,30.190000,0,0,Internet_Without_Support
8,Medium Term,6,0,1,0,High,High,108.787500,1,0,Internet_With_Support
9,Very Long Term,3,1,0,0,Medium,High,56.257258,0,0,Internet_Without_Support


In [25]:
telco_data[engineered_features].dtypes

TenureGroup            category
NumberOfServices          int64
IsDSL                     int64
IsFiberOptic              int64
NoInternet                int64
MonthlyChargesGroup         str
TotalChargesGroup           str
ChargeToTenureRatio     float64
HighValueCustomer         int64
HighRiskCustomer          int64
InternetTechSupport         str
dtype: object

In [26]:
print("Churn by Tenure Group:")
print(pd.crosstab(
    telco_data["TenureGroup"],
    telco_data["Churn"],
    normalize="index"
).round(3))

Churn by Tenure Group:
Churn              No    Yes
TenureGroup                 
New Customer    0.526  0.474
Short Term      0.713  0.287
Medium Term     0.796  0.204
Long Term       0.856  0.144
Very Long Term  0.934  0.066


In [27]:
print("\nChurn by Number of Services:")
print(pd.crosstab(
    telco_data["NumberOfServices"],
    telco_data["Churn"],
    normalize="index"
).round(3))


Churn by Number of Services:
Churn                No    Yes
NumberOfServices              
1                 0.792  0.208
2                 0.655  0.345
3                 0.623  0.377
4                 0.687  0.313
5                 0.743  0.257
6                 0.783  0.217
7                 0.883  0.117
8                 0.947  0.053


In [28]:
print("\nChurn by High-Risk Customer:")
print(pd.crosstab(
    telco_data["HighRiskCustomer"],
    telco_data["Churn"],
    normalize="index"
).round(3))


Churn by High-Risk Customer:
Churn                No    Yes
HighRiskCustomer              
0                 0.794  0.206
1                 0.308  0.692


In [30]:
telco_data.to_csv("../data/processed/feature_engineered_customers.csv", index=False)
print("csv saved")

csv saved
